# TP1

Objectifs :
- Comprendre comment la fréquence d'échantillonnage, la proportion de valeurs censurées et la proportion de valeurs aberrantes peut affecter les performances de l'algorithme
- Appliquer le modèle à différents jeux de données et devenir autonome sur le traitement de ces jeux de données

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
folder_name = 'Tunis_F2/Day_1/src/'
root_dir = '/content/gdrive/My Drive/'
base_dir = root_dir + folder_name

In [ ]:
pip install scienceplots

In [ ]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

current_dir = os.getcwd()
module_path = os.path.abspath(os.path.join(base_dir, '..', 'libs'))
if module_path not in sys.path:
    sys.path.append(module_path)

from SCOU_extended_3_1 import *
from utils import *

In [ ]:
base_dir

In [ ]:
usetex = False
use_scienceplots = False

In [ ]:
data = pd.read_csv(base_dir.split('src')[0] + 'data/dataset.csv', sep=";")
data = data.rename(columns={'scou_muX':'muX'})
data.dateStart = pd.to_datetime(data.dateStart)

In [ ]:
data.head(3)

In [ ]:
for col in ['X_t', 'obs', 'lod']:
    data[col] = np.log10(data[col])

In [ ]:
data.head(3)

In [ ]:
plot_results(data, usetex=usetex, use_scienceplots=use_scienceplots)

### Exercice 1

- Exécutez simplement les cellules suivantes. Le modèle est appliqué à un jeu de données classique, avec un échantillon par jour.

In [ ]:
ex1_data = data.copy()

In [ ]:
NB_CHAINS = 1
TUNING_ITERS = 500
SAMPLING_ITERS = 500

In [ ]:
ex1_scou = model_fit(ex1_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex1_data, ex1_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex1_data, usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

### Exercice 2

- La quantité d'observations est désormais réduite à une fréquence plus conventionnelle (deux prélèvements par semaine)

In [ ]:
ex2_data = data.copy()

In [ ]:
ex2_data['weekday'] = ex2_data.dateStart.dt.weekday

In [ ]:
ex2_data.head()

In [ ]:
ex2_data.loc[ex2_data.weekday.isin([0,1,3,4,5]), 'obs'] = np.nan

In [ ]:
ex2_data.head()

In [ ]:
ex2_scou = model_fit(ex2_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex2_data, ex2_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex2_data, usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

### Exercice 3

- La quantité d'observations est désormais réduite à une fréquence plus creuse (une observation par mois)

In [ ]:
ex3_data = data.copy()

In [ ]:
ex3_data['weekday'] = ex3_data.dateStart.dt.weekday

In [ ]:
ex3_data.head()

In [ ]:
keep_those = np.arange(start=0, stop=ex3_data.shape[0], step=31).tolist()

In [ ]:
keep_those.append(ex3_data.index.tolist()[-1])

In [ ]:
remove_those = np.setdiff1d(ex3_data.index.tolist(), keep_those)

In [ ]:
ex3_data.loc[remove_those, 'obs'] = np.nan

In [ ]:
ex3_data.head()

In [ ]:
ex3_scou = model_fit(ex3_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex3_data, ex3_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex3_data, usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

- Q1 : comparez visuellement les signaux reconstruits en fonction des trois fréquences d'échantillonnage. Commentez.
- Q2 : comparez quantitativement les signaux reconstruits en fonction des trois fréquences d'échantillonnage.

### Exercice 4

- On introduit maintenant une part de censure (certains échantillons ne peuvent être quantifiés car le signal viral présent est inférieur à la limite de détection de la méthode)

In [ ]:
censored_proportion = 25

In [ ]:
ex4_data = data.copy()
lod = np.percentile(ex4_data.loc[~ex4_data.X_t.isna()].X_t.values, censored_proportion)
ex4_data['lod'] = lod
ex4_data.loc[ex4_data.obs<= ex4_data.lod, 'obs'] = ex4_data.loc[ex4_data.obs<= ex4_data.lod, 'obs']

In [ ]:
ex4_data

In [ ]:
ex4_scou = model_fit(ex4_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex4_data, ex4_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex4_data, usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

### Exercice 5

- Comme dans l'exercice 2, la quantité d'observations est à nouveau réduite à une fréquence plus conventionnelle (deux prélèvements par semaine)

In [ ]:
ex5_data = data.copy()
lod = np.percentile(ex5_data.loc[~ex5_data.X_t.isna()].X_t.values, censored_proportion)
ex5_data['lod'] = lod
ex5_data.loc[ex5_data.obs<= ex5_data.lod, 'obs'] = ex5_data.loc[ex5_data.obs<= ex5_data.lod, 'obs']

In [ ]:
ex5_data.head()

In [ ]:
ex5_data['weekday'] = ex5_data.dateStart.dt.weekday

In [ ]:
ex5_data.loc[ex5_data.weekday.isin([0,1,3,4,5]), 'obs'] = np.nan

In [ ]:
ex5_data.head()

In [ ]:
ex5_scou = model_fit(ex5_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex5_data, ex5_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex5_data, usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

### Exercice 6

- Comme dans l'exercice 3, la quantité d'observations est désormais réduite à une fréquence plus creuse (une observation par mois).

In [ ]:
ex6_data = data.copy()
lod = np.percentile(ex6_data.loc[~ex6_data.X_t.isna()].X_t.values, censored_proportion)
ex6_data['lod'] = lod
ex6_data.loc[ex6_data.obs<= ex6_data.lod, 'obs'] = ex6_data.loc[ex6_data.obs<= ex6_data.lod, 'obs']
ex6_data['weekday'] = ex6_data.dateStart.dt.weekday

In [ ]:
ex6_data.head()

In [ ]:
keep_those = np.arange(start=0, stop=ex6_data.shape[0], step=31).tolist()

In [ ]:
keep_those.append(ex6_data.index.tolist()[-1])

In [ ]:
remove_those = np.setdiff1d(ex6_data.index.tolist(), keep_those)

In [ ]:
ex6_data.loc[remove_those, 'obs'] = np.nan

In [ ]:
ex6_data.head()

In [ ]:
ex6_scou = model_fit(ex6_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex6_data, ex6_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex6_data, usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

- Q1 : comparez visuellement les signaux reconstruits en fonction des trois fréquences d'échantillonnage. Commentez.
- Q2 : comparez quantitativement les signaux reconstruits en fonction des trois fréquences d'échantillonnage.
- Q3 : comparez également comment la censure impacte chaque signal par rapport aux signaux des trois premiers exercices, qui n'en comportaient pas.

### Exercice 7

- On reprend le cas d'un échantillonnage conventionnel (deux échantillons par semaine). On va maintenant faire varier le niveau de censure progressivement, pour estimer l'impact de ce niveau de censure sur la qualité du signal reconstruit.

In [ ]:
censored_proportions = np.arange(0.0, 110.0, 10.0)
censored_proportions

In [ ]:
ex7_datas = {}
for censored_proportion in censored_proportions:

    print('============')
    print(f'Processing proportion {censored_proportion}...')
    print('============')

    ex7_data = data.copy()
    lod = np.percentile(ex7_data.loc[~ex7_data.X_t.isna()].X_t.values, censored_proportion)
    ex7_data['lod'] = lod
    ex7_data.loc[ex7_data.obs<= ex7_data.lod, 'obs'] = ex7_data.loc[ex7_data.obs<= ex7_data.lod, 'obs']
    ex7_data['weekday'] = ex7_data.dateStart.dt.weekday
    ex7_data.loc[ex7_data.weekday.isin([0,1,3,4,5]), 'obs'] = np.nan

    ex7_scou = model_fit(ex7_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

    get_results(ex7_data, ex7_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

    ex7_datas[censored_proportion] = ex7_data

In [ ]:
for key in list(ex7_datas.keys()):
    plot_results(ex7_datas[key], usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

- Q1 : comparez visuellement les signaux reconstruits en fonction des 10 seuils de censure. Commentez.
- Q2 : comparez quantitativement les signaux reconstruits en fonction des 10 seuils de censure.
- Q3 : remarquez-vous quelque chose de particulier au niveau du temps de calcul ? Commentez.

### Exercice 8

- Nous allons maintenant rajouter la dernière couche de complexité des données : les valeurs aberrantes. Pour évaluer l'impact des valeurs aberrantes, nous allons procéder de manière similaire aux exercices qui ont précédemment traité la variation progressive de la proportion de valeurs censurées.

In [ ]:
ex8_data = data.copy()

In [ ]:
RANDOM_SEED = 48

eps = 0.5
outliers_proportion = 0.05
outlier_low, outlier_high = 2.0, 3.0

# Initialisation de la graine :
np.random.seed(RANDOM_SEED)
obsAt = np.array(ex8_data.loc[~ex8_data['obs'].isna()].index.tolist())
outliers_indexes = np.random.choice(obsAt[1:-1], size=int(obsAt.shape[0]*outliers_proportion), replace=False)
#################### -------------------- Y_t -------------------- ####################
Y_t = np.ones(ex8_data.shape[0]) * np.nan
X_t = ex8_data.X_t.values
#sign = 1
for i in range(ex8_data.shape[0]):
    if i in obsAt and i not in outliers_indexes:
        Y_t[i] = ex8_data.obs.values[i]
    elif i in obsAt and i in outliers_indexes:
        coefficient = np.random.uniform(low=outlier_low, high=outlier_high)
        coefficient_sign_rand = np.random.binomial(1, 0.5)
        coefficient_sign = 1 if coefficient_sign_rand==1 else -1
        sample_distance = coefficient_sign*coefficient*eps
        Y_t[i] = X_t[i] + sample_distance

ex8_data['obs'] = Y_t
emp_prop = outliers_indexes.shape[0] / ex8_data.shape[0]

print(f'Empirical outliers proportion = {emp_prop:.4f} | A priori outliers proportion = {outliers_proportion:.4f}')

In [ ]:
plot_results(ex8_data, usetex=usetex, use_scienceplots=use_scienceplots,
             identify_outliers=True, outliers_indexes=outliers_indexes, plot_inference=False)

In [ ]:
ex8_scou = model_fit(ex8_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex8_data, ex8_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex8_data, usetex=usetex, use_scienceplots=use_scienceplots,
             identify_outliers=True, outliers_indexes=outliers_indexes, plot_inference=True)

### Exercice 9

- La quantité d'observations est désormais réduite à une fréquence plus conventionnelle (deux prélèvements par semaine)

In [ ]:
ex9_data = data.copy()

In [ ]:
ex9_data.head()

In [ ]:
ex9_data['weekday'] = ex9_data.dateStart.dt.weekday

In [ ]:
ex9_data.loc[ex9_data.weekday.isin([0,1,3,4,5]), 'obs'] = np.nan

In [ ]:
ex9_data.head()

In [ ]:
RANDOM_SEED = 48

eps = 0.5
outliers_proportion = 0.05
outlier_low, outlier_high = 2.0, 3.0

# Initialisation de la graine :
np.random.seed(RANDOM_SEED)
obsAt = np.array(ex9_data.loc[~ex9_data['obs'].isna()].index.tolist())
outliers_indexes = np.random.choice(obsAt[1:-1], size=int(obsAt.shape[0]*outliers_proportion), replace=False)
#################### -------------------- Y_t -------------------- ####################
Y_t = np.ones(ex9_data.shape[0]) * np.nan
X_t = ex9_data.X_t.values
#sign = 1
for i in range(ex9_data.shape[0]):
    if i in obsAt and i not in outliers_indexes:
        Y_t[i] = ex9_data.obs.values[i]
    elif i in obsAt and i in outliers_indexes:
        coefficient = np.random.uniform(low=outlier_low, high=outlier_high)
        coefficient_sign_rand = np.random.binomial(1, 0.5)
        coefficient_sign = 1 if coefficient_sign_rand==1 else -1
        sample_distance = coefficient_sign*coefficient*eps
        Y_t[i] = X_t[i] + sample_distance

ex9_data['obs'] = Y_t
emp_prop = outliers_indexes.shape[0] / obsAt.shape[0]

print(f'Empirical outliers proportion = {emp_prop:.4f} | A priori outliers proportion = {outliers_proportion:.4f}')

In [ ]:
plot_results(ex9_data, usetex=usetex, use_scienceplots=use_scienceplots,
             identify_outliers=True, outliers_indexes=outliers_indexes, plot_inference=False)

In [ ]:
ex9_scou = model_fit(ex9_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex9_data, ex9_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex9_data, usetex=usetex, use_scienceplots=use_scienceplots,
             identify_outliers=True, outliers_indexes=outliers_indexes, plot_inference=True)

### Exercice 10

- La quantité d'observations est désormais réduite à une fréquence plus creuse (une observation par mois)

In [ ]:
ex10_data = data.copy()

In [ ]:
ex10_data.head()

In [ ]:
keep_those = np.arange(start=0, stop=ex10_data.shape[0], step=31).tolist()

In [ ]:
keep_those.append(ex10_data.index.tolist()[-1])

In [ ]:
remove_those = np.setdiff1d(ex10_data.index.tolist(), keep_those)

In [ ]:
ex10_data.loc[remove_those, 'obs'] = np.nan

In [ ]:
ex10_data.head()

In [ ]:
RANDOM_SEED = 48

eps = 0.5
outliers_proportion = 0.05
outlier_low, outlier_high = 2.0, 3.0

# Initialisation de la graine :
np.random.seed(RANDOM_SEED)
obsAt = np.array(ex10_data.loc[~ex10_data['obs'].isna()].index.tolist())
outliers_indexes = np.random.choice(obsAt[1:-1], size=int(obsAt.shape[0]*outliers_proportion), replace=False)
#################### -------------------- Y_t -------------------- ####################
Y_t = np.ones(ex10_data.shape[0]) * np.nan
X_t = ex10_data.X_t.values
#sign = 1
for i in range(ex10_data.shape[0]):
    if i in obsAt and i not in outliers_indexes:
        Y_t[i] = ex10_data.obs.values[i]
    elif i in obsAt and i in outliers_indexes:
        coefficient = np.random.uniform(low=outlier_low, high=outlier_high)
        coefficient_sign_rand = np.random.binomial(1, 0.5)
        coefficient_sign = 1 if coefficient_sign_rand==1 else -1
        sample_distance = coefficient_sign*coefficient*eps
        Y_t[i] = X_t[i] + sample_distance

ex10_data['obs'] = Y_t
emp_prop = outliers_indexes.shape[0] / obsAt.shape[0]

print(f'Empirical outliers proportion = {emp_prop:.4f} | A priori outliers proportion = {outliers_proportion:.4f}')

In [ ]:
plot_results(ex10_data, usetex=usetex, use_scienceplots=use_scienceplots,
             identify_outliers=True, outliers_indexes=outliers_indexes, plot_inference=False)

In [ ]:
ex10_scou = model_fit(ex10_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

In [ ]:
get_results(ex10_data, ex10_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

In [ ]:
plot_results(ex10_data, usetex=usetex, use_scienceplots=use_scienceplots,
             identify_outliers=True, outliers_indexes=outliers_indexes, plot_inference=True)

- Quelles métriques pouvez-vous utiliser pour évaluer les performances de l'algorithme dans la détection de valeurs abérrantes ?

### Exercice 11

- On reprend le cas d'un échantillonnage conventionnel (deux échantillons par semaine). On va maintenant faire varier le niveau de censure progressivement, pour estimer l'impact de ce niveau de censure sur la qualité du signal reconstruit.

In [ ]:
outliers_proportions = np.arange(0.0, 14.0, 2.0)
outliers_proportions = outliers_proportions/100
outliers_proportions

In [ ]:
ex11_datas = {}
for outliers_proportion in outliers_proportions:

    print('============')
    print(f'Processing proportion {outliers_proportion}...')
    print('============')

    RANDOM_SEED = 48
    outlier_low, outlier_high = 2.0, 3.0

    ex11_data = data.copy()
    ex11_data['weekday'] = ex11_data.dateStart.dt.weekday
    ex11_data.loc[ex11_data.weekday.isin([0,1,3,4,5]), 'obs'] = np.nan

    # Initialisation de la graine :
    np.random.seed(RANDOM_SEED)
    obsAt = np.array(ex11_data.loc[~ex11_data['obs'].isna()].index.tolist())
    outliers_indexes = np.random.choice(obsAt[1:-1], size=int(obsAt.shape[0]*outliers_proportion), replace=False)
    #################### -------------------- Y_t -------------------- ####################
    Y_t = np.ones(ex11_data.shape[0]) * np.nan
    X_t = ex11_data.X_t.values
    #sign = 1
    for i in range(ex11_data.shape[0]):
        if i in obsAt and i not in outliers_indexes:
            Y_t[i] = ex11_data.obs.values[i]
        elif i in obsAt and i in outliers_indexes:
            coefficient = np.random.uniform(low=outlier_low, high=outlier_high)
            coefficient_sign_rand = np.random.binomial(1, 0.5)
            coefficient_sign = 1 if coefficient_sign_rand==1 else -1
            sample_distance = coefficient_sign*coefficient*eps
            Y_t[i] = X_t[i] + sample_distance

    ex11_data['obs'] = Y_t
    emp_prop = outliers_indexes.shape[0] / obsAt.shape[0]

    print(f'Empirical outliers proportion = {emp_prop:.4f} | A priori outliers proportion = {outliers_proportion:.4f}')

    ex11_scou = model_fit(ex11_data, 'obs', 'lod', NB_CHAINS=NB_CHAINS, TUNING_ITERS=TUNING_ITERS, SAMPLING_ITERS=SAMPLING_ITERS)

    get_results(ex11_data, ex11_scou, remove_those = [], NB_CHAINS=NB_CHAINS)

    ex11_datas[outliers_proportion] = ex11_data

In [ ]:
for idx, key in enumerate(list(ex11_datas.keys())):

    print('========')
    print(f'Results for pout = {outliers_proportions[idx]} :')
    print('========')

    plot_results(ex11_datas[key], usetex=usetex, use_scienceplots=use_scienceplots, plot_inference=True)

- Q1 : comparez visuellement les signaux reconstruits en fonction des 10 seuils de censure. Commentez.
- Q2 : comparez quantitativement les signaux reconstruits en fonction des 10 seuils de censure.
- Q3 : remarquez-vous quelque chose de particulier au niveau du temps de calcul ? Commentez.

### Exercice 12

C'est à vous ! Vous allez appliquer le modèle de lissage au jeu de données suivant. Vous avez à votre disposition les débits mesurés le jour j (plantVolume) et le débit journalier moyen annualisé (meanVolume) pour prendre en compte les effets de dilution.

- Q1 : traitez chaque station du jeu de données en suivant le protocole de traitement
- Q2 : trouvez un moyen de proposer un signal aggrégé pour l'ensemble de l'agglomération parisienne


- BONUS :
    - Q3 : trouvez un moyen de comparer les niveaux de circulation entre deux stations de cette agglomération, tout en prenant en compte les effets de dilution
    - Q4 : trouvez un moyen de comparer les tendances de ces stations
    - Q5 : trouvez un moyen de comparer la circulation dans les eaux usées aux cas cliniques (fichier PARIS.csv)

In [ ]:
db = pd.read_csv(base_dir.split('src')[0] + 'data/paris_dataset.csv', sep=";")

In [ ]:
db.head()

### Exercice 13

- Récupérez le jeu de données hébergé à l'adresse suivante : https://data.mendeley.com/datasets/733pw8jfwb/1

- Q1 : Traitez les données de SARS-CoV-2 en ne considérant que les concentrations brutes
- Q2 : Même question, mais corrigez les effets de dilution en utilisant les informations de débit
- Q3 : Même question, mais corrigez les effets de dilution en utilisant les informations des crAssphages
- Q4 : Même question, mais corrigez les effets de dilution en utilisant les informations du PMMoV

### Exercice 14

- Application à votre propre fichier de données, si vous en avez un à disposition